# Identify and Extract Job Ads in Historical Newspapers

## Get candidate pages from iiif manifest

In [1]:
import pandas as pd
import json
import requests
import PIL
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import torch
from tqdm.notebook import trange, tqdm
import time
import shutil
import os
from io import BytesIO
import re

In [2]:
iiif_df = pd.read_csv("data_dh-infra.csv")

In [3]:
url = iiif_df['iiif_manifest'][0]

In [8]:
images_list = []
id_list = []
for index, row in tqdm(iiif_df.iterrows()):
    url = row['iiif_manifest']
    try:
        response = requests.get(url)
        data = response.json()
        for entry in data.get("metadata", []):
            label_de = entry.get("label", {}).get("de", [""])[0]
        
            if label_de == "Jahr/Datierung":
                edition_date = entry.get("value", {}).get("de", [""])[0]
                edition_year = pd.to_datetime(edition_date, dayfirst=True).strftime('%Y')
                edition_date = pd.to_datetime(edition_date, dayfirst=True).strftime('%d-%m-%Y')
                break
    
        images = [anno["body"]["id"] for canvas in data.get("items", []) for page in canvas.get("items", []) for anno in page.get("items", [])
                  if anno.get("motivation") == "painting"]
        images_id = [iiif_df['anno_id'][0]+f'_{i}' for i in range(0, len(images))]
        images_list.append(images)
        id_list.append(images_id)
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the URL: {e}")
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
page_df = pd.DataFrame({'anno_id': id_list, 'iiif_manifest': images_list})

## Check if candidate pages contain any job ads 

In [14]:
from transformers import AutoModelForImageClassification, AutoImageProcessor
from torch import nn

model_repo = 'Var3n/DiT_classification_jobads_finetuned'
image_processor = AutoImageProcessor.from_pretrained(model_repo)
model = AutoModelForImageClassification.from_pretrained(model_repo)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


BeitForImageClassification(
  (beit): BeitModel(
    (embeddings): BeitEmbeddings(
      (patch_embeddings): BeitPatchEmbeddings(
        (projection): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): BeitEncoder(
      (layer): ModuleList(
        (0): BeitLayer(
          (attention): BeitAttention(
            (attention): BeitSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=False)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): BeitSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): BeitIntermediate(
            (dense): Lin

In [15]:
os.makedirs('Backup', exist_ok=True)

In [ ]:
images = page_df['iiif_manifest'].to_list()

In [16]:
n = 25 
img_batches = [images[i:i+n] for i in range(0, len(images), n)]
id_batches = [images_id[i:i+n] for i in range(0, len(images_id), n)]
back_up = 0
for ind, batch in tqdm(enumerate(img_batches), total=len(img_batches)):
    id_batch = id_batches[ind]
    images_data = []
    preds = []
    probs = []
    restart = False 
    retry = 0
    for image in batch:
        while True:
            try:
                img_data = requests.get(image)
                img_data = PIL.Image.open(BytesIO(img_data.content)).convert('RGB')
                images_data.append(img_data)
                if restart:
                    t = time.localtime()
                    current_time = time.strftime('%d-%m %H:%M:%S', t)
                    restart = False
                    retry = 0
                    print(f'{current_time}: Reconnected')
            except OSError as e1:
                t = time.localtime()
                current_time = time.strftime('%d-%m %H:%M:%S', t)
                print(f'{current_time}: {e1} \n Rate limit. Retry after 1 minute.')
                time.sleep(60)
                restart = True
                if retry == 3:
                    print(f'Tried {retry} times. Skipping image.')
                    break
                retry += 1
                continue
            else:
                break
    encoding = image_processor(images_data, return_tensors = 'pt')
    encoding = encoding.to(device)

    with torch.no_grad():
        outputs = model(**encoding)
        logits = outputs.logits
    preds.extend(logits.argmax(-1).tolist())
    probs.extend(nn.functional.softmax(outputs.logits, dim = -1).amax(-1).tolist())
    with open('all_ads.txt', 'a') as f:
        for (image_id, image, pred, prob) in zip(id_batch, batch, preds, probs):
            f.write('{0} {1} {2} {3}\n'.format(image_id, image, pred, prob))
    del encoding
    torch.cuda.empty_cache()
    back_up += 1
    if back_up == 100:
        now = datetime.now()
        dest = 'Backup/all_ads_'+now.strftime('%d_%m_%Y_%H_%M')+'.txt'
        shutil.copy('all_ads.txt', dest)
        back_up = 0

  0%|          | 0/1 [00:00<?, ?it/s]

## Layout Analysis 

## OCR (needs testing for parallel processing once Layout is done)

In [56]:
import os
import glob
from concurrent.futures import ProcessPoolExecutor
from tqdm.contrib.concurrent import process_map
from PIL import Image, ImageDraw
import tesserocr

# Import OCR-D core tools
from ocrd_models.ocrd_page import parse, TextEquivType, to_xml

In [3]:
# Global variable to hold the API instance for each worker process
worker_tess_api = None

In [4]:
def initialize_worker(tessdata_path, model_name):
    """
    Runs once per CPU core when the worker process starts.
    Loads the Tesseract model into memory for this specific process.
    """
    global worker_tess_api
    worker_tess_api = tesserocr.PyTessBaseAPI(
        path=tessdata_path, 
        lang=model_name, 
        psm=tesserocr.PSM.RAW_LINE
    )

In [54]:
def process_single_pagexml(args):
    """
    The main function executed by the workers. 
    It extracts the image path directly from the XML.
    """
    xml_path, output_xml_path, image_base_dir = args
    global worker_tess_api
    worker_id = multiprocessing.current_process()._identity[0] - 1
    filename = os.path.basename(xml_path)
    
    # 1. Parse XML
    pcgts = parse(xml_path, silence=True)
    page = pcgts.get_Page()
    
    # 2. Dynamically get the image path from the PageXML
    rel_image_path = page.get_imageFilename() 
    image_path = os.path.join(image_base_dir, rel_image_path)
    
    if not os.path.exists(image_path):
        return f"Error: Image not found at {image_path} (XML: {xml_path})"
        
    image = Image.open(image_path).convert('RGB')
    
    # 3. Iterate over text lines and mask polygons
    regions = page.get_AllRegions(classes=['Text'])
    all_lines = [line for region in regions for line in region.get_TextLine()]
    for region in regions:
        for line in region.get_TextLine():
            line.set_TextEquiv([])
            
            coords_string = line.get_Coords().points
            polygon = [(int(x), int(y)) for x, y in (pt.split(',') for pt in coords_string.split())]
            if not polygon:
                continue
            
            # Crop image to bounding box
            xs, ys = zip(*polygon)
            left, top, right, bottom = min(xs), min(ys), max(xs), max(ys)
            bbox_image = image.crop((left, top, right, bottom))
            
            # Mask adjacent text using the polygon
            shifted_polygon = [(x - left, y - top) for x, y in polygon]
            mask = Image.new('L', bbox_image.size, 0)
            ImageDraw.Draw(mask).polygon(shifted_polygon, outline=255, fill=255)
            
            white_bg = Image.new('RGB', bbox_image.size, (255, 255, 255))
            line_image = Image.composite(bbox_image, white_bg, mask)
            
            # 4. Run Tesseract using the worker's globally initialized API
            worker_tess_api.SetImage(line_image)
            text = worker_tess_api.GetUTF8Text().strip()
            conf = worker_tess_api.MeanTextConf() / 100.0 
            
            if text:
                line.add_TextEquiv(TextEquivType(Unicode=text, conf=conf))

    # 5. Save the updated XML
    with open(output_xml_path, "w", encoding="utf-8") as f:
        f.write(to_xml(pcgts))
    
    return f"Successfully processed: {os.path.basename(xml_path)}"

In [62]:
def run_parallel_ocr(xml_files, output_dir, image_base_dir, tessdata_path, model_name, max_workers=4):
    """
    Manages the pool of workers.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Bundle arguments for the map function: (xml_in, xml_out, image_base_directory)
    args_list = []
    for xml_path in xml_files:
        filename = os.path.basename(xml_path)
        output_xml_path = os.path.join(output_dir, filename)
        args_list.append((xml_path, output_xml_path, image_base_dir))

    # Create the pool and distribute the workload
    with ProcessPoolExecutor(max_workers=max_workers, initializer=initialize_worker, initargs=(tessdata_path, model_name)) as executor:
        
        # Map the function, and wrap the resulting generator in tqdm.notebook
        results = list(tqdm(executor.map(process_single_pagexml, args_list),total=len(args_list),desc="Processing XML files"))
        
    # Print any errors that occurred during processing
    for result in results:
        if "Error" in result:
            print(result)

In [ ]:
rootdir = '/home/lab6/nextcloud/Jobads'

In [63]:
run_parallel_ocr(xml_files, f'{rootdir}/ocr_out', rootdir, f'/usr/share/tesseract-ocr/5/tessdata', 'frak2021-0.905')

Processing XML files:   0%|          | 0/1 [00:00<?, ?it/s]

### Adjust file names of pngs enhanced by Eynollah

In [26]:
rootdir = '/home/lab6/job_ads'
enhan_imgs = glob.glob(f'{rootdir}/*.png')
xml_files = glob.glob(f'{rootdir}/*.xml')

In [10]:
for img in tqdm(enhan_imgs):
    new_name = re.sub(r'_enhanced', '', img)
    os.rename(img, new_name)

  0%|          | 0/1 [00:00<?, ?it/s]

### Change file paths inside xml files to enhanced png

In [44]:
pcgts = parse(xml_files[0], silence=True)
page = pcgts.get_Page()
old_filename = page.get_imageFilename()
new_filename = old_filename.replace("/home/lab6/job_ads/nfp/1886/", "")
new_filename = new_filename.replace(".jpg", ".png")
page.set_imageFilename(new_filename)

with open(xml_files[0], "w", encoding="utf-8") as f:
    f.write(to_xml(pcgts))

## Identify Job Ad regions

In [1]:
from bs4 import BeautifulSoup
import joblib
from transformers import AutoTokenizer
import torch
from transformers import AutoModelForSequenceClassification

In [ ]:
tag = 'btb'

In [ ]:
rootdir = '/home/lab6/nextcloud/Jobads'

In [ ]:
paths = glob.glob(f"{rootdir}/final_selection/{tag}_ocr/????/*.xml")

In [ ]:
# models still need to be uploaded to huggingface
model = AutoModelForSequenceClassification.from_pretrained("C:/Users/venglaro/Documents/job_ads_project/job_ads_identification/results_zeitungs/best")

# Load tokenizer and put model in evaluation mode
tokenizer = AutoTokenizer.from_pretrained("stefan-it/zeitungs-lm-v1")
model.eval()

# Choose device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
results = []

for path in tqdm(paths):
    path = os.path.normpath(path)
    xml_content = None

    for attempt in range(3):
        try:
            with open(path, "r", encoding="utf-8") as file:
                xml_content = file.read()
            break
        except (OSError, ValueError) as e:
            time.sleep(0.2)

    if xml_content is None:
        print(f"❌ Failed to read: {path}")
        continue  # skip this file entirely

    soup = BeautifulSoup(xml_content, "xml")
    text_regions = soup.find_all("pc:TextRegion")
    
    # Iterate through each text region and classify
    for region in text_regions:
        region_id = region.get("id", "Unknown ID")  # Get the region ID

        # Extract the last TextEquiv (if available)
        text_equivs = region.find_all("TextEquiv")
        text_content = text_equivs[-1].Unicode.text if text_equivs and text_equivs[-1].Unicode else "No text"

        # Extract the coordinates of the region (assuming we have min/max x and y coordinates)
        coords = region.find("Coords").get('points')
        coords_list = coords.split()
        coords_list = [tuple(map(int, point.split(','))) for point in coords_list]

        # Calculate the min/max x and y for cropping
        x_coords = [coord[0] for coord in coords_list]
        y_coords = [coord[1] for coord in coords_list]

        min_x = min(x_coords)
        max_x = max(x_coords)
        min_y = min(y_coords)
        max_y = max(y_coords)

        if len(text_content) > 20:        
            inputs = tokenizer(text_content, return_tensors="pt", truncation=True, padding=True)

            # Move to same device as the model
            inputs = {k: v.to(device) for k, v in inputs.items()}

            # Disable gradient calculation
            with torch.no_grad():
                outputs = model(**inputs)
                logits = outputs.logits
                probs = torch.softmax(logits, dim=1)
                pred_class = torch.argmax(probs, dim=1).item()  # 0 or 1

            prediction = pred_class

            # If prediction is 1 (Job Ad), save the region and coordinates
            if prediction == 1:
                
                filename = os.path.basename(path)
                

                # Store the region details (ID, text, coordinates) in the results list
                results.append({"metadata": filename[8:-4], "region_id": region_id, "text": text_content, "min_x": min_x, "max_x": max_x, 
                                "min_y": min_y, "max_y": max_y})

In [ ]:
df = pd.DataFrame(results)

In [ ]:
df.to_csv(f"{tag}_classified.csv", index=False)

## Post-Correction

In [ ]:
tag = 'btb'

In [ ]:
def get_byt_len(text):
    return len(tokenizer(text, return_tensors="pt").input_ids[0])

In [ ]:
path = f"{tag}_classified.csv"

In [ ]:
df['post-corrected'] = None

In [ ]:
df['text'] = df['text'].str.replace(r'ﬁ', r'fi', regex=True)
df['text'] = df['text'].str.replace(r'ﬂ', r'fl', regex=True)
df['text'] = df['text'].str.replace(r'¬', r'-', regex=True)
df['text'] = df['text'].str.replace(r'¶', r'', regex=True)
df['text'] = df['text'].str.replace(r'˖', r'+', regex=True)
df['text'] = df['text'].str.replace(r'‟', r'"', regex=True)
df['text'] = df['text'].str.replace(r'⅙', r'1/6', regex=True)
df['text'] = df['text'].str.replace(r'ﬀ', r'ff', regex=True)
df['text'] = df['text'].str.replace(r'ꝓ', r'ff', regex=True)
df['text'] = df['text'].str.replace(r'ů', r'u', regex=True)
df['text'] = df['text'].str.replace(r'̃', r'', regex=True)
df['text'] = df['text'].str.replace(r'̈', r'', regex=True)
df['text'] = df['text'].str.replace(r'ﬄ', r'ffl', regex=True)
df['text'] = df['text'].str.replace(r'⸗', r'-', regex=True)
df['text'] = df['text'].str.replace(r'ꝛ', r'r', regex=True)
for vok, uml in [('a', 'ä'), ('o', 'ö'), ('u', 'ü')]:
    df['text'] = df['text'].str.replace(vok+r'ͤ', uml, regex=True)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("hmbyt5-preliminary/byt5-small-historic-multilingual-span20-flax")
tokenizer.model_max_length=150
model = AutoModelForSeq2SeqLM.from_pretrained("C:/Users/venglaro/Downloads/hmbyt5_frak_correction_100_adafactor_150/hmbyt5_frak_correction_100_adafactor_150").to("cuda")

In [ ]:
df['text_byt_len'] = df['text'].apply(get_byt_len)

In [ ]:
new_text_list = []
for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    new_text = []
    if row['text_byt_len'] > 150:
        text_split = row['text'].split()
        if len(text_split) == 1:
            temp_text = row['text']
            text_half_1, text_half_2 = temp_text[:round(len(temp_text)/2)], temp_text[round(len(temp_text)/2):]
            new_text.extend([text_half_1, text_half_2])
        else:
            split_idx = round(len(text_split)/2)
        
            text_half_1 = text_split[:split_idx]
            text_half_1 = " ".join(text_half_1)
            text_half_2 = text_split[split_idx:]
            text_half_2 = " ".join(text_half_2)

            new_text.extend([text_half_1, text_half_2])
        
    else:
        new_text.append(row['text'])
    
    new_text_list.append(new_text)

In [ ]:
df['split_text'] = new_text_list
df = df.explode(['split_text']).reset_index(drop=False)

In [ ]:
df['text_byt_len'] = df['split_text'].apply(get_byt_len)

In [ ]:
df.sort_values(by=["text_byt_len"])

In [ ]:
max_byt_len = df['text_byt_len'].max()

while max_byt_len > 150:
    new_text_list = []
    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        new_text = []
        if row['text_byt_len'] > 150:
            text_split = row['split_text'].split()
            if len(text_split) == 1:
                temp_text = row['split_text']
                text_half_1, text_half_2 = temp_text[:round(len(temp_text)/2)], temp_text[round(len(temp_text)/2):]
                new_text.extend([text_half_1, text_half_2])
            else:
                split_idx = round(len(text_split)/2)
                text_half_1 = text_split[:split_idx]
                text_half_1 = " ".join(text_half_1)
                text_half_2 = text_split[split_idx:]
                text_half_2 = " ".join(text_half_2)
                new_text.extend([text_half_1, text_half_2])
        else:
            new_text.append(row['split_text'])
        new_text_list.append(new_text)
    df['split_text'] = new_text_list
    df = df.explode(['split_text']).reset_index(drop=True)
    df['text_byt_len'] = df['split_text'].apply(get_byt_len)
    if df['text_byt_len'].max() == max_byt_len:
        print("ERROR: byt_len can't be reduced further!")
        break
    max_byt_len = df['text_byt_len'].max()
    print(f'current max_byt_len = {max_byt_len}')

In [ ]:
df.sort_values(by=["text_byt_len"])

In [ ]:
preds = df['split_text'].to_list()
n = 130 # adjust this number for optimal GPU usage (e.g. check task manager: Dedicated GPU memory usage should be almost full during post_correction but shared memory should not be used)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id
tokenizer.padding_side = "left"
pred_chunk = [preds[i:i + n] for i in range(0, len(preds), n)]
res = []

for chunk in tqdm(pred_chunk):
    input = chunk
    input = tokenizer.batch_encode_plus(input, return_tensors="pt", padding=True)#.input_ids
    input = input.to("cuda")
    output = model.generate(**input, max_new_tokens=len(input.input_ids[0])+5, num_beams=4, do_sample=True)
    temp_res = tokenizer.batch_decode(output, skip_special_tokens=True)
    temp_res = [s.replace("<pad>", "") for s in temp_res]
    res.extend(temp_res)
    del input
    torch.cuda.empty_cache()

In [ ]:
df['post-corrected'] = res
df['post-corrected'] = df['post-corrected'].replace(r"^ +| +$", r"", regex=True)

In [ ]:
comb_df = df.groupby("index",as_index=False)['post-corrected'].agg(lambda x: ' '.join(x))

In [ ]:
df = df.drop_duplicates(subset=["index"])

In [ ]:
df.reset_index(drop=True, inplace = True)
df.drop(columns=["index"], inplace = True)

In [ ]:
df['post-corrected'] = comb_df['post-corrected']

In [ ]:
df.to_csv(f"df_{tag}_postcorrected.csv", index=False)

## Identify Job Ad Class

In [ ]:
tag = 'btb'

In [ ]:
df = pd.read_csv(f"df_{tag}_postcorrected.csv")

In [ ]:
from transformers import ElectraForSequenceClassification, ElectraTokenizer

load_dir = "C:/Users/venglaro/Documents/job_ads_project/job_ads_classification/electra_finetuned"

model = ElectraForSequenceClassification.from_pretrained(load_dir)
tokenizer = ElectraTokenizer.from_pretrained(load_dir)

device = 'cuda'

model.to(device)
model.eval()   # important for inference

In [ ]:
predictions = []

for text_content in df["post-corrected"].astype(str):
    inputs = tokenizer(
        text_content,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        pred_class = torch.argmax(logits, dim=1).item()

    predictions.append(pred_class)

In [ ]:
import pickle

with open("C:/Users/venglaro/Documents/job_ads_project/job_ads_classification/electra_finetuned/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

In [ ]:
df["prediction"] = predictions

# decode back to original labels
df["prediction_label"] = label_encoder.inverse_transform(df["prediction"])

In [ ]:
df.to_csv(f"df_{tag}_postcorrected_classified.csv", index=False)

## Extract Job Information

In [ ]:
import spacy

In [ ]:
tag = 'btb'

In [ ]:
df = pd.read_csv(f"df_{tag}_postcorrected_classified.csv")

In [ ]:
nlp_salary = spacy.load("C:/Users/venglaro/Documents/job_ads_project/ner/model-best")
nlp_positions = spacy.load("C:/Users/venglaro/Documents/job_ads_project/NLP4DH/ner/model-best_GPU")
nlp_requirements = spacy.load("C:/Users/venglaro/Documents/job_ads_project/requirements/model-best")

In [ ]:
# Function to extract entities with offsets
def extract_entities(model, text):
    doc = model(text)
    entities = [[ent.start_char, ent.end_char, ent.label_] for ent in doc.ents]
    return entities if entities else None

In [ ]:
def show_ner_labels(nlp, name):
    labels = nlp.get_pipe("ner").labels
    print(f"\n{name} ({len(labels)} labels):")
    for label in labels:
        print(f"  - {label}")

show_ner_labels(nlp_salary, "Salary model")
show_ner_labels(nlp_positions, "Positions model")
show_ner_labels(nlp_requirements, "Requirements model")

In [ ]:
salary_labels = nlp_salary.get_pipe("ner").labels
print(salary_labels)

In [ ]:
for label in salary_labels:
    df[f"{label}"] = None

In [ ]:
def extract_ner_to_columns(nlp, text):
    doc = nlp(text)
    entities_by_label = {}

    for ent in doc.ents:
        offset = (ent.start_char, ent.end_char)
        entities_by_label.setdefault(ent.label_, []).append(offset)

    return entities_by_label

In [ ]:
for idx, text in df["post-corrected"].astype(str).items():
    ents = extract_ner_to_columns(nlp_salary, text)

    for label, values in ents.items():
        df.at[idx, f"{label}"] = values

In [ ]:
position_labels = nlp_positions.get_pipe("ner").labels
print(position_labels)

In [ ]:
for label in position_labels:
    df[f"{label}"] = None

In [ ]:
for idx, text in df["post-corrected"].astype(str).items():
    ents = extract_ner_to_columns(nlp_positions, text)

    for label, values in ents.items():
        df.at[idx, f"{label}"] = values

In [ ]:
requirements_labels = nlp_requirements.get_pipe("ner").labels
print(requirements_labels)

In [ ]:
for label in requirements_labels:
    df[f"{label}"] = None

In [ ]:
for idx, text in df["post-corrected"].astype(str).items():
    ents = extract_ner_to_columns(nlp_requirements, text)

    for label, values in ents.items():
        df.at[idx, f"{label}"] = values

In [ ]:
df.to_csv(f"df_{tag}_extracted_entities.csv", index=False)